# 기억잇다 · 그림일기 판정기

**세 가지 실행 방식**: `train_and_evaluate`(학습·성능 비교), `evaluate_only`(기존 가중치 평가), `predict_diaries`(앱에서 내보낸 그림 판정).

BGE-M3 한국어 인코더와 SigLIP2 그림 인코더를 동결하고 결합·3분류 모듈만 학습합니다. 실제 연구용 가중치·성능 결과는 동봉하지 않았습니다.

- GPU와 Internet을 켭니다. 코드 ZIP과 실제 사람 검토 JSON을 비공개 Dataset으로 연결합니다.
- 학습·평가에는 사람이 항목별 정답을 붙인 그림이 필요합니다. 없는 경우 앱에서 자료 수집부터 시작합니다.
- 단순 평균 결합 / 패치 어텐션 / 어텐션+관계 대조 학습을 같은 데이터·시드로 비교합니다.
- API 비교는 별도 선택이며 기본은 유료 호출 0회입니다.
- Kaggle의 localhost는 노트북 안의 주소입니다. 앱과 연결할 때는 결과 JSON을 가져오거나 가중치를 내려받아 로컬 판정 서버를 실행합니다.


In [ ]:
from pathlib import Path
import os, sys, json, zipfile, subprocess
RUN_ROOT=Path('/kaggle/working/memory-verifier'); RUN_ROOT.mkdir(parents=True,exist_ok=True)
INPUT_ROOT=Path('/kaggle/input')
MODE="train_and_evaluate"  # train_and_evaluate / evaluate_only / predict_diaries
PROJECT_ZIP=""             # 여러 코드 ZIP이 연결됐으면 경로 지정
PROJECT_PATH=""            # 이미 압축 해제된 프로젝트면 경로 지정
EXPORT_DIR=INPUT_ROOT        # 사람 검토 JSON 위치
DATASET_DIR=""             # 이미 준비한 train/dev/test.jsonl 폴더, 없으면 자동 준비
CHECKPOINTS={}             # evaluate_only: {"relation":"/kaggle/input/.../relation-seed42"}
BATCH_CHECKPOINT=""         # predict_diaries: 학습한 가중치 폴더
REQUEST_DIR=INPUT_ROOT       # 앱의 Kaggle 판정 요청 JSON 위치
RUN_NAME="experiment-v1"   # 새 실험은 새 이름 사용, 기존 결과를 덮어쓰지 않음
TEXT_MODE="sentence"       # tokens는 별도 비교 실험
SEEDS=[42]                  # 정식 반복 실험 예: [42,43,44]
EPOCHS=15
BATCH_SIZE=8
MEASURE_END_TO_END=False     # True: 인코더 포함 시간, False: 캐시 특징+판정 모듈 시간
API_MODEL=""               # 실제 사용 가능한 모델 ID/스냅샷 지정
RUN_PAID_API=False
assert MODE in {"train_and_evaluate","evaluate_only","predict_diaries"}
if PROJECT_PATH:
    PROJECT=Path(PROJECT_PATH)
else:
    candidates=[]
    if PROJECT_ZIP:candidates=[Path(PROJECT_ZIP)]
    else:
        for archive_path in INPUT_ROOT.rglob("*.zip"):
            try:
                with zipfile.ZipFile(archive_path) as archive:
                    if any(n.endswith("/ml/diary_verifier/fusion_train.py") or n=="ml/diary_verifier/fusion_train.py" for n in archive.namelist()):candidates.append(archive_path)
            except zipfile.BadZipFile:pass
    if candidates:
        if len(candidates)!=1:raise RuntimeError("PROJECT_ZIP에 사용할 코드 ZIP 하나를 지정하세요.")
        parent=RUN_ROOT/"project";parent.mkdir(exist_ok=True)
        with zipfile.ZipFile(candidates[0]) as archive:
            if any(not (parent/n).resolve().is_relative_to(parent.resolve()) or ((item.external_attr>>16)&0o170000)==0o120000 for item in archive.infolist() for n in [item.filename]):raise RuntimeError("안전하지 않은 ZIP 경로입니다.")
            archive.extractall(parent)
        roots=[p.parent.parent.parent for p in parent.rglob("diary_verifier/fusion_train.py")]
    else:roots=[p.parent.parent.parent for p in INPUT_ROOT.rglob("diary_verifier/fusion_train.py")]
    if len(roots)!=1:raise RuntimeError("판정기 코드 ZIP을 Dataset으로 연결하거나 PROJECT_PATH를 지정하세요.")
    PROJECT=roots[0]
assert (PROJECT/"ml/diary_verifier/fusion_train.py").is_file()
os.environ["PYTHONPATH"]=str(PROJECT/"ml");sys.path.insert(0,str(PROJECT/"ml"))
os.environ["HF_HOME"]=str(RUN_ROOT/"hf-cache")
def run(*args):subprocess.run([sys.executable,"-m",*map(str,args)],check=True,env=os.environ.copy())
print("코드:",PROJECT,"실행 방식:",MODE)


In [ ]:
subprocess.run([sys.executable,"-m","pip","install","-r",str(PROJECT/"ml/requirements-fusion.txt")],check=True)
import torch
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch",torch.__version__,"실행 장치",DEVICE)
if torch.cuda.is_available():print(torch.cuda.get_device_name(0))


## 사람 정답 자료와 분리

앱에서 그림 생성 → `사람 정답 붙이기 · 학습자료 만들기` → 각 항목에 직접 판정 → 검토 저장 → 실험 JSON 내보내기.

AI 판정은 사람 정답으로 자동 복사하지 않습니다. 예시/출처 미확인 기록은 아래에서 제외하고 제외 수를 표시합니다. 불완전한 사람 판정은 중단합니다. 고정 시험 자료는 독립 복수 검토를 권장합니다.

같은 사건·그림·원본/변형본·prompt family는 같은 그룹에 넣습니다. 70/15/15는 사건 묶음 기준입니다. 최소 3개 그룹은 실행 조건이며 충분한 연구 표본 수를 뜻하지 않습니다.


In [ ]:
DATA=None
if MODE!="predict_diaries":
    if DATASET_DIR:DATA=Path(DATASET_DIR)
    else:
        reviewed=[];excluded=0
        for path in EXPORT_DIR.rglob("*.json"):
            try:value=json.loads(path.read_text(encoding="utf-8"))
            except (ValueError,UnicodeError,OSError):continue
            if not isinstance(value,dict) or not isinstance(value.get("records"),list):continue
            for record in value["records"]:
                if record.get("humanSource")=="manual_review" and record.get("imageProvenance",{}).get("kind") in {"generated_api","uploaded_generated"}:reviewed.append(record)
                else:excluded+=1
        print("사람 검토 기록",len(reviewed),"제외한 예시/출처 미확인 기록",excluded)
        if not reviewed:raise RuntimeError("사람 정답이 붙은 실제 생성 그림이 없습니다. 앱에서 자료를 수집해 연결하세요. 임의 정확도를 생성하지 않습니다.")
        export=RUN_ROOT/"human-reviewed-only.json";export.write_text(json.dumps({"records":reviewed},ensure_ascii=False),encoding="utf-8")
        DATA=RUN_ROOT/("data-"+RUN_NAME)
        if (DATA/"dataset-info.json").exists():raise RuntimeError("이 이름의 데이터가 이미 있습니다. DATASET_DIR로 재사용하거나 RUN_NAME을 바꾸세요.")
        run("diary_verifier.prepare",export,"--out",DATA)
    for name in ["train.jsonl","dev.jsonl","test.jsonl","dataset-info.json"]:
        if not (DATA/name).is_file():raise RuntimeError("준비된 데이터 파일이 누락되었습니다: "+name)
    print((DATA/"dataset-info.json").read_text())


## 인코더 특징 추출

사람 정답을 제외한 확인 장면·인물 구별 정보·사실 하나를 인코딩합니다. 텍스트/그림 인코더 버전과 입력 해시를 고정합니다. 특징 캐시가 같은 차원이어도 모델 버전이 다르면 재사용하지 않습니다.


In [ ]:
FEATURES=RUN_ROOT/("features-"+RUN_NAME)
if MODE!="predict_diaries":
    options=[]
    if MODE=="evaluate_only":
        if not CHECKPOINTS:raise RuntimeError("CHECKPOINTS에 평가할 학습 가중치 폴더를 지정하세요.")
        configs=[json.loads((Path(path)/"fusion_config.json").read_text()) for path in CHECKPOINTS.values()]
        if any(c["encoders"]!=configs[0]["encoders"] for c in configs):raise RuntimeError("이 실행에서는 동일 인코더의 가중치끼리 비교하세요. 인코더가 다르면 별도 특징 캐시를 준비합니다.")
        bundle=configs[0]["encoders"];FEATURES.mkdir(exist_ok=True)
        existing=FEATURES/"encoders.json"
        if existing.exists() and json.loads(existing.read_text())!=bundle:raise RuntimeError("새 특징 캐시 이름이 필요합니다.")
        existing.write_text(json.dumps(bundle),encoding="utf-8")
        options=["--text-model",bundle["text"]["model_id"],"--image-model",bundle["image"]["model_id"],"--pooling",bundle["text"]["pooling"],"--max-tokens",str(bundle["text"]["max_tokens"])]
    manifests=[DATA/"test.jsonl"] if MODE=="evaluate_only" else [DATA/(n+".jsonl") for n in ["train","dev","test"]]
    run("diary_verifier.fusion_features","--manifests",*manifests,"--cache",FEATURES,"--device",DEVICE,*options)


## 학습 및 동일 시험 자료 비교

A: 그림 패치를 평균한 단순 결합 모델. B: 사실 문장과 그림 패치의 어텐션 모델. C: B와 같은 구조에 동일 사실의 정상/오류 그림 쌍 대조 학습을 추가한 모델.

같은 인코더·자료·시드·배치 조건으로 학습하며 dev loss로 체크포인트를 선택합니다. 쌍이 없는 경우 C는 실행하지 않고 생략 사유를 기록합니다. 학습을 위해 인코더를 미세조정하는 코드는 아니며 결합·판정 모듈을 학습합니다.


In [ ]:
OUT=RUN_ROOT/RUN_NAME
predictions=[]
if MODE=="train_and_evaluate":
    args=["--data",DATA,"--cache",FEATURES,"--out",OUT,"--device",DEVICE,"--epochs",str(EPOCHS),"--batch-size",str(BATCH_SIZE),"--seeds",*map(str,SEEDS),"--text-mode",TEXT_MODE]
    if MEASURE_END_TO_END:args.append("--end-to-end")
    run("diary_verifier.experiment",*args)
    predictions=sorted(OUT.glob("*.jsonl"))
elif MODE=="evaluate_only":
    OUT.mkdir(exist_ok=True)
    for name,checkpoint in CHECKPOINTS.items():
        if not name or Path(name).name!=name:raise RuntimeError("비교 이름에는 폴더 경로를 쓰지 마세요.")
        target=OUT/(name+".jsonl")
        args=["--manifest",DATA/"test.jsonl","--checkpoint",checkpoint,"--out",target,"--device",DEVICE]
        args += ["--from-images"] if MEASURE_END_TO_END else ["--cache",FEATURES]
        run("diary_verifier.fusion_predict",*args);predictions.append(target)
    run("diary_verifier.evaluate","--manifest",DATA/"test.jsonl","--predictions",*predictions,"--out",OUT/"comparison.json")
if MODE!="predict_diaries":
    print("성능표 HTML/CSV/JSON:",OUT)
    from IPython.display import display,HTML
    for path in sorted(OUT.glob("comparison*.html")):display(HTML(path.read_text(encoding="utf-8")))


## 범용 API 비교 — 선택

정답을 제외한 동일 그림·장면·인물·사실을 제공합니다. 사용할 수 있는 강한 모델과 정확한 버전을 지정하세요. `RUN_PAID_API=False`이면 실제 호출하지 않습니다. API 실패/누락 결과를 정상으로 대체하지 않습니다.

캐시 판정 시간과 API 전체 응답 시간을 직접 비교하지 마세요. 시간 비교는 `MEASURE_END_TO_END=True`로 실행하고 환경·단위·호출 수를 함께 기록하세요.


In [ ]:
if MODE!="predict_diaries" and API_MODEL:
    run("diary_verifier.api_baseline","--manifest",DATA/"test.jsonl","--model",API_MODEL,"--out",OUT/"api.jsonl","--dry-run")
    if RUN_PAID_API:
        try:
            from kaggle_secrets import UserSecretsClient
            api_key=UserSecretsClient().get_secret("DIARY_API_KEY")
        except Exception:
            import getpass
            api_key=getpass.getpass("API key: ")
        os.environ["DIARY_API_KEY"]=api_key
        try:run("diary_verifier.api_baseline","--manifest",DATA/"test.jsonl","--model",API_MODEL,"--out",OUT/"api.jsonl")
        finally:os.environ.pop("DIARY_API_KEY",None);del api_key
        run("diary_verifier.evaluate","--manifest",DATA/"test.jsonl","--predictions",*predictions,OUT/"api.jsonl","--out",OUT/"comparison-with-api.json")


## 앱의 그림을 Kaggle에서 판정 — predict_diaries

앱의 완성 화면 → `판정기 연결 · Kaggle 자료` → `Kaggle 판정 요청 저장`.

요청 JSON과 학습 가중치를 Dataset으로 연결하고 위의 MODE/BATCH_CHECKPOINT를 설정합니다. 아래에서 만든 `diary-predictions.json`을 내려받아 앱의 `Kaggle 결과 불러오기`로 가져옵니다. 그림·사실 목록 해시가 달라지면 연결을 거부합니다. 이 모드는 사람 정답이 없어도 추론할 수 있지만 정확도는 계산하지 않습니다.


In [ ]:
if MODE=="predict_diaries":
    if not BATCH_CHECKPOINT:raise RuntimeError("BATCH_CHECKPOINT에 실제 사람 정답으로 학습한 가중치 폴더를 지정하세요.")
    requests=[]
    for path in REQUEST_DIR.rglob("*.json"):
        try:value=json.loads(path.read_text(encoding="utf-8"))
        except (ValueError,UnicodeError,OSError):continue
        if isinstance(value,dict) and value.get("kind")=="memory_connect_verification_requests":requests.append(path)
    if not requests:raise RuntimeError("앱에서 내보낸 Kaggle 판정 요청 JSON을 연결하세요.")
    OUT.mkdir(exist_ok=True)
    run("diary_verifier.batch_verify","--requests",*requests,"--checkpoint",BATCH_CHECKPOINT,"--out",OUT/"diary-predictions.json","--device",DEVICE)
    print("앱에서 불러올 결과:",OUT/"diary-predictions.json")


## 결과 내려받기와 실시간 앱 연결

아래 ZIP에는 판정 모듈 가중치·설정·예측·성능표를 넣습니다. 원본 이미지·API 키·거대한 인코더 캐시는 포함하지 않습니다. Kaggle Output을 저장하고 ZIP/JSON을 내려받으세요.

실시간 앱 연결은 내려받은 가중치를 로컬에서 실행합니다:

```bash
PYTHONPATH=ml python -m diary_verifier.fusion_serve --checkpoint <학습 폴더> --port 8010
```

앱 `.env`: `DIARY_VERIFY_BACKEND=local`, `DIARY_VERIFIER_URL=http://127.0.0.1:8010`, `DIARY_VERIFY_AUTO=true`.

그림 생성 → 사실별 자동 판정 → 결과와 함께 저장됩니다. 서버 미연결·오류는 미판정으로 남고 범용 API로 자동 대체하지 않습니다. 기존 기록은 유지합니다.


In [ ]:
RESULT=RUN_ROOT/(RUN_NAME+"-results.zip")
with zipfile.ZipFile(RESULT,"w",zipfile.ZIP_DEFLATED) as archive:
    for path in OUT.rglob("*"):
        if path.is_file() and path.suffix in {".pt",".json",".jsonl",".csv",".html",".md"}:archive.write(path,path.relative_to(OUT))
    if DATA is not None:archive.write(DATA/"dataset-info.json","dataset-info.json")
print("다운로드할 결과 ZIP:",RESULT)
